# Weighted-embedding feedback: decoding without ever choosing a token

Standard decoding commits. At each step it picks one token and feeds that
token's embedding back in. This notebook does not pick. It feeds back

$$e_t = \sum_{v \in V} p_t(v)\, E[v]$$

the probability-weighted average of **every** row of the input embedding table.
The model therefore conditions on a continuous vector that is not any real
token, and the trajectory never commits to text at all.

**There is no beam search here, and that is not an omission.** The mixed vector
is a deterministic function of the prefix, so there is nothing to branch on:
every hypothesis sharing a prefix would receive an identical input and stay
identical. One trajectory per configuration is the whole search.

**What varies.** Sixty configurations per model:

| axis | values |
|---|---|
| temperature | 0.25, 0.5, 1.0, 2.0, 4.0 |
| mixture | whole vocabulary, or the top 5 renormalised |
| magnitude | fed raw, or rescaled to the mean token-embedding norm |
| history | keep every mixture, commit each once passed, or keep only the newest |

plus a hard-decoded greedy run as the control, in the same batch and the same
session, so divergence is measured step by step rather than across runs.

**The prompt is the same as the beam-search notebook's**, deliberately, so the
two experiments are comparable. Neither states a length: the token budget is
enforced by the loop.

<details>
<summary>Why the magnitude axis exists</summary>

A convex combination of vectors pointing in different directions is shorter
than any of them, and gets shorter as the distribution flattens. At temperature
4 the fed vector is both pointing somewhere no token points *and* too small.
Running raw and rescaled separates "wrong direction" from "wrong magnitude" -
without it, any degradation is uninterpretable.
</details>

<details>
<summary>Why the history axis exists</summary>

With a **soft** history nothing is ever committed: at step 50 the model is
attending over forty-nine continuous vectors, none of which any token could
have produced, so error compounds and the question being asked is *can a model
survive a context made entirely of mixtures?*

With a **committed** history each position is replaced by the embedding of the
token its own distribution named, as soon as the loop moves past it. Exactly
one position - the newest - is ever continuous, so the question narrows to *can
a model absorb a single off-manifold vector?* Any degradation that survives
committing is caused by one soft step; anything that disappears was caused by
accumulation.

With a **last-only** history the context is thrown away entirely: the model is
handed the newest mixture and nothing else, not even the prompt. That asks the
sharpest version of the question - *does the mixed vector carry enough on its
own to continue?* - and makes the trajectory a Markov chain on the embedding
space.

The greedy control needs no history variant. Its mixture is already one-hot on
the argmax, so committing rewrites a vector to itself.
</details>

## 0. Runtime

**Set the runtime first:** *Runtime > Change runtime type > A100 GPU*. Which GPU
you get decides how many of the seven models run at all - the fit check in
section 3 prints the verdict per model before anything downloads.


In [ ]:
# Colab ships torch with CUDA already; only the HF stack needs topping up.
# Pinning nothing on purpose: these architectures are new enough that the
# newest transformers is the one most likely to know about them.
%pip install -q -U transformers accelerate safetensors huggingface_hub

import gc
import json
import math
import os
import re
import shutil
import time
from pathlib import Path

import pandas as pd
import torch
import transformers

SEED = 0
torch.manual_seed(SEED)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

if torch.cuda.is_available():
    DEVICE = "cuda"
    _props = torch.cuda.get_device_properties(0)
    GPU_NAME = _props.name
    VRAM_GIB = _props.total_memory / 2**30
    # bfloat16 needs Ampere or newer. A T4 (Turing) has no bf16 support at all,
    # so fall back to float16 there. On a GPU float16 is fine - the "float16 is
    # slower" rule in this project is a CPU rule and does not apply here.
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DEVICE, GPU_NAME, VRAM_GIB, DTYPE = "cpu", "none", 0.0, torch.float32

print(f"device   {DEVICE}  ({GPU_NAME})")
print(f"VRAM     {VRAM_GIB:.1f} GiB")
print(f"dtype    {DTYPE}")
print(f"torch    {torch.__version__}")
print(f"transformers {transformers.__version__}")

if DEVICE == "cpu":
    print("\nWARNING: no GPU. Runtime > Change runtime type > GPU, then rerun.")


### Where results go

Mounting Drive is what makes this sweep survivable. It runs for an hour or more
and downloads ~127 GiB; a runtime recycled at model six would otherwise take
every finished result with it. With results on Drive, re-running section 6 skips
whatever already completed and carries on.

**The parent folder is this notebook's own filename**, not a typed constant, so
a second experiment in a second notebook lands beside this one rather than on
top of it - the same reason `results_dir()` derives the per-model folder from
the model id. Neither way of asking Colab for the notebook name is a documented
API, so if both fail the notebook says so loudly and you set `EXPERIMENT` by
hand rather than writing into a folder named after nothing.

Set `USE_DRIVE = False` for a throwaway run.

In [ ]:
# Colab wipes /content when the runtime is recycled, and this sweep runs for an
# hour or more across seven models. Results therefore go to Drive, so a
# disconnect costs only the model in flight rather than everything finished so
# far - and section 6 can resume instead of starting over.
#
# Weights deliberately do NOT go to Drive: ~127 GiB in total would exhaust the
# quota, and Drive is far slower to write than local scratch. They stay on the
# local disk and are deleted after each model.
USE_DRIVE = True


def notebook_name(fallback=""):
    """This notebook's filename stem, so each notebook keeps its own results.

    Derived rather than typed, for the same reason `results_dir()` derives the
    per-model folder: a hard-coded name means a second notebook silently writes
    into this one's results. Asked two ways because neither is a documented,
    supported API - the sessions endpoint is Colab-internal, and the frontend
    request needs a live browser connection, so either can stop working.
    """
    try:
        import requests

        sessions = requests.get(
            "http://172.28.0.12:9000/api/sessions", timeout=5).json()
        if sessions and sessions[0].get("name"):
            return Path(sessions[0]["name"]).stem
    except Exception:                   # noqa: BLE001 - any failure means "try the next way"
        pass

    try:
        from google.colab import _message

        ipynb = _message.blocking_request("get_ipynb", timeout_sec=30)
        name = ipynb["ipynb"]["metadata"].get("colab", {}).get("name", "")
        if name:
            return Path(name).stem
    except Exception:                   # noqa: BLE001 - same
        pass

    return fallback


# Set EXPERIMENT by hand to override the detected name.
EXPERIMENT = re.sub(r"[^A-Za-z0-9._-]+", "_", notebook_name()).strip("_")

if not EXPERIMENT:
    raise RuntimeError(
        "could not read this notebook's name from Colab, and results must not "
        "go to a folder named after nothing - a second notebook would overwrite "
        "this one's results. Set EXPERIMENT = \"<name>\" in this cell and re-run."
    )

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = Path("/content/drive/MyDrive") / EXPERIMENT
else:
    RESULTS_ROOT = Path("/content/research/results") / EXPERIMENT

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(f"experiment  {EXPERIMENT}")
print(f"results ->  {RESULTS_ROOT}")
print("  survives a runtime restart" if USE_DRIVE
      else "  EPHEMERAL - lost if the runtime is recycled")

## 1. Infrastructure, inlined

Locally these helpers live in `src/research/` and the notebooks import them.
There is no `research` package here: the repo has no git remote to install from,
and its `pyproject.toml` pins Python 3.13 and routes `torch` to a **CPU-only**
index - which on a GPU box would silently install a CPU build and quietly defeat
the whole exercise.

So the handful of helpers the experiment needs are reproduced below. `blocks()`
is the same structural search as the local version - the longest `ModuleList`
whose children share a class - not a hard-coded module path, so it survives all
seven naming conventions.


In [ ]:
# --- infrastructure, inlined ------------------------------------------------
# Locally this lives in src/research/ and notebooks import it. There is no
# `research` package on Colab (the repo has no git remote to pip-install from,
# and it pins python 3.13 plus a CPU-only torch index, both wrong here), so the
# few helpers the experiment needs are reproduced below. Behaviour matches the
# local versions; `blocks()` in particular is the same structural search, not a
# hard-coded module path.

ROOT = Path("/content/research")
MODELS_DIR = ROOT / "models"          # HF cache; local scratch, wiped per model
RESULTS = RESULTS_ROOT                # set in the Drive cell above
for _d in (MODELS_DIR, RESULTS):
    _d.mkdir(parents=True, exist_ok=True)

# Point the HF cache at our own folder before huggingface_hub is imported
# anywhere that matters, same intent as research/__init__.py. `os` is imported
# in the environment cell above; what matters here is that this assignment runs
# before the huggingface_hub import on the next line.
os.environ["HF_HUB_CACHE"] = str(MODELS_DIR)

from huggingface_hub import snapshot_download  # noqa: E402
from transformers import AutoTokenizer  # noqa: E402


def model_slug(model_id: str) -> str:
    """google/gemma-4-12B -> google_gemma-4-12B. Filesystem-safe folder name."""
    slug = re.sub(r"[^A-Za-z0-9._-]+", "_", str(model_id).strip()).strip("_")
    if not slug:
        raise ValueError(f"cannot derive a folder name from {model_id!r}")
    return slug


def results_dir(model_id: str) -> Path:
    path = RESULTS / model_slug(model_id)
    path.mkdir(parents=True, exist_ok=True)
    return path


def blocks(model):
    """The repeated decoder blocks, whatever this architecture calls them.

    The longest ModuleList whose entries all share one class. Structural, so it
    works across transformer.h / model.layers / gpt_neox.layers without knowing
    which one this model uses.
    """
    best = []
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.ModuleList) or len(module) < 2:
            continue
        if len({type(child) for child in module}) != 1:
            continue
        if len(module) > len(best):
            best = [(f"{name}.{i}", child) for i, child in enumerate(module)]
    if not best:
        raise LookupError("could not find a stack of decoder blocks")
    return best


def block_names(model):
    return [name for name, _ in blocks(model)]


def repo_dir(model_id: str) -> Path:
    return MODELS_DIR / f"models--{model_id.replace('/', '--')}"


def disk_free_gib() -> float:
    return shutil.disk_usage("/content").free / 2**30


def download(model_id: str) -> str:
    """Fetch weights, skipping the ONNX/GGUF mirrors some of these repos ship."""
    return snapshot_download(
        model_id,
        cache_dir=MODELS_DIR,
        allow_patterns=["*.safetensors", "*.safetensors.index.json",
                        "*.json", "*.txt", "*.model", "*.jinja"],
        ignore_patterns=["onnx/*", "*.gguf", "*.onnx", "*.pth", "*.bin"],
    )


def auto_classes():
    """Loader classes to try, in order. Not every name exists in every version.

    `AutoModelForCausalLM` is not enough on its own. A model that generates text
    but also accepts images is registered against a vision class instead, and
    asking the wrong Auto class produces "Unrecognized configuration class ...
    for this kind of AutoModel" - which is what skipped Ministral-3-14B on the
    first run, despite it being a perfectly loadable text generator.
    """
    names = ["AutoModelForCausalLM", "AutoModelForImageTextToText",
             "AutoModelForVision2Seq", "AutoModelForSeq2SeqLM"]
    return [(n, getattr(transformers, n)) for n in names if hasattr(transformers, n)]


def load(model_id: str, trust: bool = False):
    """Load onto the GPU at DTYPE. Returns (model, tokenizer).

    `trust` is per model rather than global: only the repos whose registry entry
    says they need custom modelling code get it, so enabling it for DeepSeek
    does not silently enable it for everything else.
    """
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, cache_dir=MODELS_DIR, trust_remote_code=trust)

    tried = []
    for name, cls in auto_classes():
        try:
            model = cls.from_pretrained(
                model_id, cache_dir=MODELS_DIR, dtype=DTYPE,
                device_map=DEVICE, trust_remote_code=trust)
        except ValueError as exc:
            # Only the "wrong Auto class" error is worth trying the next class
            # for. Anything else - out of memory, a missing file, a refused
            # trust_remote_code - is a real failure and must not be swallowed.
            if "Unrecognized configuration class" not in str(exc):
                raise
            tried.append(name)
            continue
        if tried:
            print(f"  (loaded via {name}; {', '.join(tried)} did not accept it)", flush=True)
        model.eval()
        return model, tokenizer

    raise ValueError(
        f"no AutoModel class accepted {model_id}; tried {', '.join(tried)}"
    )


def build_prompt(tokenizer, instruction):
    """Render the instruction the way this model expects it - if it says how.

    Returns ``(text, used_template)``.

    The project rule is never to *assume* a chat template but to ask the
    tokenizer, which is what this does. It matters here: the prompt is an
    instruction, and feeding an instruction to an instruction-tuned model as raw
    completion text is why the first run produced continuations like
    `The user says: "Write a funny story..."` - the model was completing a
    transcript rather than answering.

    Base models have no chat template and correctly fall back to the raw string.
    Which happened is recorded per model, because it changes what the numbers
    mean.
    """
    template = getattr(tokenizer, "chat_template", None)
    if not (USE_CHAT_TEMPLATE and template):
        return instruction, False

    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": instruction}],
        tokenize=False,
        add_generation_prompt=True,      # start the assistant turn, don't just quote the user
    )

    # Many templates embed the BOS token, and the tokenizer will add another one
    # when the search tokenises this string. A doubled BOS is not an error the
    # model will report - it is simply a prompt it was never trained on - so drop
    # the duplicate, but only once it is confirmed the tokenizer really does
    # prepend one.
    bos = getattr(tokenizer, "bos_token", None)
    bos_id = getattr(tokenizer, "bos_token_id", None)
    if bos and bos_id is not None and text.startswith(bos):
        probe = tokenizer("x")["input_ids"]
        if probe[:1] == [bos_id]:
            text = text[len(bos):]

    return text, True


def vocab_size(model):
    """Vocabulary size, wherever this architecture keeps it.

    Multimodal configs nest the text settings under `text_config` and leave
    `vocab_size` unset at the top level - which is why gemma-4-12B reported a
    blank vocabulary on the first run.
    """
    for probe in (model.config, getattr(model.config, "text_config", None)):
        size = getattr(probe, "vocab_size", None) if probe is not None else None
        if size:
            return int(size)
    embeddings = model.get_output_embeddings()
    if embeddings is not None:
        return int(embeddings.weight.shape[0])
    raise LookupError(f"cannot determine vocab_size for {type(model).__name__}")


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def delete_weights(model_id: str):
    """Remove one model's weights from disk.

    Six of these models are 112 GiB of downloads between them, against a Colab
    disk of roughly 110-235 GiB. Unloading from VRAM is not enough - the files
    have to go too, or the run dies on disk part-way through.
    """
    path = repo_dir(model_id)
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)


## 2. The models

**Text-only language models, and that is the point.** An earlier version of this
sweep mixed in four vision-language models, and it went wrong in ways that were
invisible until the results came back:

- `gemma-4-12B` spent its whole 100-token budget emitting `<image|>` placeholder
  tokens. Given no image, the empty image slot dominated its next-token
  distribution - the search was correct, the setup was not.
- `Ministral-3-14B` would not load at all: it registers as
  `Mistral3ForConditionalGeneration`, which `AutoModelForCausalLM` refuses.
- `blocks()` finds the longest stack of identical modules, and on a VLM
  checkpoint that can be the **vision tower** rather than the text decoder - so
  the reported depth may not describe the thing being searched.

Every model below is a `*ForCausalLM` with no vision tower, no image tokens and
no processor config, so none of that can happen. `bf16_GiB` comes from the
safetensors headers on the Hub rather than an estimate.

Six mechanisms across seven models: Mamba-2 at two different SSM-to-attention
ratios, short convolutions, Compressed Convolutional Attention, Multi-head
Latent Attention, and plain dense attention twice - once as a GQA baseline and
once as base weights for the control.

**One mechanism is lost and cannot be recovered here: Gated DeltaNet.** Every
Qwen3.5 checkpoint is a vision model, and Qwen3-Next, which carries GDN, is 80B.
`Qwen/Qwen3-8B` is dense GQA - a baseline, not a substitute.


In [ ]:
# Every model, in the order they run. `bf16_GiB` is measured from the
# safetensors headers on the Hub, not estimated: it is the number the fit check
# below compares against real VRAM.
# Text-only language models, smallest first. Every entry is a `*ForCausalLM`
# with no vision tower, which is what keeps this comparison clean - see the
# markdown above for what the multimodal versions cost.
#
# None needs `trust_remote_code`: transformers recognises all seven config
# classes natively, so no repo's own Python is executed here.
MODELS = [
    {"id": "ibm-granite/granite-4.0-h-tiny",   "params_B":  6.94, "bf16_GiB": 12.9,
     "remote_code": False,
     "family": "Mamba-2 heavy (~9:1 SSM:attention) + MoE"},
    {"id": "tiiuae/Falcon-H1R-7B",             "params_B":  7.59, "bf16_GiB": 14.1,
     "remote_code": False,
     "family": "attention + Mamba-2 SSM interleaved, dense"},
    {"id": "Qwen/Qwen3-8B",                    "params_B":  8.19, "bf16_GiB": 15.3,
     "remote_code": False,
     "family": "dense transformer + grouped-query attention"},
    {"id": "LiquidAI/LFM2.5-8B-A1B",           "params_B":  8.47, "bf16_GiB": 15.8,
     "remote_code": False,
     "family": "short-conv LIV + GQA, sparse MoE (~1.5B active)"},
    {"id": "Zyphra/Zaya1-8B",                  "params_B":  8.84, "bf16_GiB": 16.5,
     "remote_code": False,
     "family": "MoE + Compressed Convolutional Attention (~760M active)"},
    {"id": "mistralai/Mistral-Nemo-Base-2407", "params_B": 12.25, "bf16_GiB": 22.8,
     "remote_code": False,
     "family": "plain dense transformer, base weights - the control"},
    {"id": "deepseek-ai/DeepSeek-V2-Lite",     "params_B": 15.71, "bf16_GiB": 29.3,
     "remote_code": False,
     "family": "Multi-head Latent Attention + DeepSeekMoE (64 experts, 6 active)"},
]

# --- what was dropped, and why -------------------------------------------
#
# Vision-language models. Half the previous shortlist turned out to carry a
# vision tower, which showed up in the results rather than in the plan:
# gemma-4-12B beam-searched its way into emitting <image|> placeholder tokens
# with no image to describe, and Ministral-3 would not load through
# AutoModelForCausalLM at all because it registers as ForConditionalGeneration.
#
#   Qwen/Qwen3.5-9B                    Qwen3_5ForConditionalGeneration
#   google/gemma-4-12B                 Gemma4UnifiedForConditionalGeneration
#   mistralai/Ministral-3-14B-...      Mistral3ForConditionalGeneration
#   moonshotai/Kimi-VL-A3B-Thinking    KimiVLForConditionalGeneration
#
# Too large for one A100, at any precision. Still analysable from safetensors
# headers without downloading - see scripts/compare_architectures.py:
#
#   moonshotai/Kimi-Linear-48B-A3B-Base   49.12 B    91.5 GiB
#   deepseek-ai/DeepSeek-V4-Flash        158.07 B   294.4 GiB
#
# One mechanism is lost by going text-only and cannot be recovered in this size
# band: **Gated DeltaNet**. Every Qwen3.5 checkpoint (9B, 4B, 2B) is a vision
# model, and Qwen3-Next - which does carry GDN - is 80B. Qwen/Qwen3-8B above is
# dense GQA, so it is a strong baseline rather than a replacement mechanism.

pd.DataFrame(MODELS)[["id", "params_B", "bf16_GiB", "family"]]


## 3. Configuration, and what will actually fit

The search itself is 100 steps, one trajectory per configuration, no early stop:
**EOS is recorded but never acted on**, so every trajectory runs the full budget
and the rows stay aligned for a step-by-step comparison. What a model does after
it wanted to stop is part of what is being measured.

The token budget lives here and nowhere else - the prompt asks for a story and
says nothing about length.

The fit check is unchanged from the beam notebook: weights must sit in VRAM with
room for activations and the logits tensor, so the budget is total VRAM minus a
headroom allowance. An A100 runs all seven; DeepSeek-V2-Lite at 29.3 GiB is the
ceiling.

In [ ]:
PROMPT = "Write a funny story about a teddy bear."

# Same prompt as naive-beam-search-colab.ipynb, and it must stay that way: the
# point of this notebook is a comparison against that one, and a prompt that
# differs by so much as a word makes the two incomparable.

USE_CHAT_TEMPLATE = True

TEMPERATURES = [0.25, 0.5, 1.0, 2.0, 4.0]
MIXTURES = ["full", "top5"]      # whole vocabulary, or top 5 renormalised
MAGNITUDES = ["raw", "rescaled"]  # as computed, or scaled to mean embedding norm
HISTORIES = ["soft", "committed", "last-only"]  # see section 4 for what each means

CONFIG = dict(
    num_tokens_to_generate=100,  # hard stop: the loop runs this many steps,
                                 # whatever the model intends
    record_top_k=10,             # how many tokens to record per step
    mixture_top_k=5,             # how many the truncated mixture sums over
    device=None,                 # None = follow the model. It is on the GPU.
)

# No KV cache, for the same reason as the beam notebook: several of these models
# are hybrid SSM/attention, and a cache that works for three of seven is a
# confound rather than an optimisation. It costs O(n^2) in generated length.
#
# Here there is a second reason. The fed vector is continuous, so there is no
# token id to append - each step re-runs the model over an `inputs_embeds`
# tensor that grew by one row. Nothing about that is cacheable across steps in a
# way that would survive the architectures involved.

TRUST_REMOTE_CODE = False

VRAM_HEADROOM_GIB = 2.5
VRAM_BUDGET_GIB = max(VRAM_GIB - VRAM_HEADROOM_GIB, 0.0)

print(f"VRAM budget for weights: {VRAM_BUDGET_GIB:.1f} GiB "
      f"({VRAM_GIB:.1f} total - {VRAM_HEADROOM_GIB} headroom)")
print(f"disk free: {disk_free_gib():.0f} GiB\n")

runnable = 0
for m in MODELS:
    if m["bf16_GiB"] > VRAM_BUDGET_GIB:
        verdict = f"SKIP - needs {m['bf16_GiB']:.1f} GiB"
    elif m.get("remote_code") and not TRUST_REMOTE_CODE:
        verdict = "SKIP - needs TRUST_REMOTE_CODE"
    else:
        verdict = "runs"
        runnable += 1
    print(f"  {m['id']:45s} {m['bf16_GiB']:6.1f} GiB   {verdict}")

print(f"\n{runnable} of {len(MODELS)} will run")
rows = len(TEMPERATURES) * len(MIXTURES) * len(MAGNITUDES) * len(HISTORIES)
print(f"{rows} soft trajectories + 1 greedy control per model, all in one batch")

## 4. The implementation

Three cells, tagged `weighted-embedding-implementation` so `tests/` executes the
notebook's own code rather than a copy of it.

The whole idea is one substitution. Ordinary decoding does

```
next_id = pick(p)                 # argmax, sample, whatever
inputs = cat([inputs, E[next_id]])
```

and this does

```
inputs = cat([inputs, p @ E])     # nothing is picked
```

Everything else - the configuration axes, the instrumentation - exists to make
that substitution measurable.

### 4a. Implementation, part 1 of 3

In [ ]:
# Imported here as well as in the setup cell, so the implementation cells below
# stand on their own if lifted out of this notebook.
from dataclasses import dataclass, field
from inspect import Parameter, signature

import torch

GREEDY = "greedy"        # the control: a one-hot "mixture" over the argmax
SOFT = "soft"            # every generated position keeps its mixed vector
COMMITTED = "committed"  # each position becomes its argmax once the loop passes it
LAST_ONLY = "last-only"  # the context is the newest mixture and nothing else


@dataclass
class DecodeConfig:
    """One trajectory: how to weight the vocabulary, and how to scale the result."""

    name: str
    temperature: float
    mixture: str            # "full" | "top5" | GREEDY
    rescale: bool
    history: str = SOFT     # SOFT | COMMITTED

    @property
    def is_control(self):
        return self.mixture == GREEDY


@dataclass
class WeightedEmbeddingResult:
    """What the run produced, as plain Python - hand it to pandas or print it."""

    prompt: str
    prompt_token_ids: list
    config: dict
    configs: list = field(default_factory=list)     # one DecodeConfig per row
    steps: list = field(default_factory=list)       # per (config, step) scalars
    top_tokens: list = field(default_factory=list)  # per (config, step, rank)
    readouts: dict = field(default_factory=dict)    # config name -> argmax ids
    embedding_scale: float = 1.0                    # measured, not assumed
    scale_errors: dict = field(default_factory=dict)


def build_configs(temperatures, mixtures, magnitudes, histories=(SOFT,)):
    """The full cross of the three axes, plus the greedy control as the last row.

    The control is a configuration rather than a separate run so it shares the
    batch, the prompt and the step index with everything it is compared against.
    Divergence is then a per-step comparison between rows, not between runs.
    """
    configs = [
        DecodeConfig(name=f"T{temperature:g}-{mixture}-{magnitude}-{history}",
                     temperature=temperature, mixture=mixture,
                     rescale=(magnitude == "rescaled"), history=history)
        for temperature in temperatures
        for mixture in mixtures
        for magnitude in magnitudes
        for history in histories
    ]
    # The control gets no history variant: its mixture is already one-hot on the
    # argmax, so committing would rewrite a vector to itself.
    configs.append(DecodeConfig(name=GREEDY, temperature=1.0,
                                mixture=GREEDY, rescale=False, history=SOFT))
    return configs


def validate_configs(configs, num_tokens_to_generate, record_top_k, mixture_top_k):
    """Reject configurations that cannot mean what they say, before any compute."""
    if num_tokens_to_generate < 1:
        raise ValueError("num_tokens_to_generate must be at least 1")
    if record_top_k < 1:
        raise ValueError("record_top_k must be at least 1")
    if mixture_top_k < 1:
        raise ValueError("mixture_top_k must be at least 1")
    if not configs:
        raise ValueError("no configurations to run")
    names = [c.name for c in configs]
    if len(set(names)) != len(names):
        raise ValueError(f"configuration names must be unique: {names}")
    for config in configs:
        if not config.temperature > 0:
            raise ValueError(f"{config.name}: temperature must be positive")
        if config.mixture not in ("full", "top5", GREEDY):
            raise ValueError(f"{config.name}: unknown mixture {config.mixture!r}")
        if config.history not in (SOFT, COMMITTED, LAST_ONLY):
            raise ValueError(f"{config.name}: unknown history {config.history!r}")


def resolve_device(model, requested):
    """Where to build the tensors: the model's own device, unless told otherwise."""
    model_device = next(model.parameters()).device
    if requested is None:
        return model_device
    device = torch.device(requested)
    if device.type != model_device.type:
        raise ValueError(
            f"device={requested!r} but the model is on {model_device}. "
            "Move the model first; this function does not relocate it."
        )
    return model_device


def logits_to_keep_arg(model):
    """Name of the forward argument that trims logits to the last position, if any.

    Passing 1 makes the model run the LM head for the final position only,
    turning a (configs, length, vocab) tensor into (configs, 1, vocab) - the
    largest single allocation in the loop.
    """
    parameters = signature(model.forward).parameters
    for name in ("logits_to_keep", "num_logits_to_keep"):
        parameter = parameters.get(name)
        if parameter is not None and parameter.kind is not Parameter.VAR_KEYWORD:
            return name
    return None


def eos_token_ids(model, tokenizer):
    """Every id that ends a sequence, asking the model first and the tokenizer second.

    Recorded but never acted on: every trajectory runs the full token budget so
    the rows stay step-aligned.
    """
    raw = getattr(getattr(model, "generation_config", None), "eos_token_id", None)
    if raw is None:
        raw = getattr(tokenizer, "eos_token_id", None)
    if raw is None:
        return set()
    if isinstance(raw, int):
        return {raw}
    return {int(token_id) for token_id in raw if token_id is not None}

### 4b. Implementation, part 2 of 3

In [ ]:
def mixture_weights(probs, configs, mixture_top_k):
    """Turn each row's distribution into the weights that row feeds back.

    ``probs`` is (configs, vocab) and each row is handled according to its own
    configuration, so all of them advance in one batch:

    - ``full``   - the distribution unchanged: every token contributes.
    - ``top5``   - all but the top `mixture_top_k` zeroed, then renormalised, so
                   the vector stays in the hull of a few real embeddings.
    - ``greedy`` - one-hot on the argmax, which is ordinary hard decoding
                   expressed in the same arithmetic. It is also the limit the
                   soft rows approach as temperature goes to zero, which makes
                   it the right control rather than merely a familiar one.
    """
    weights = torch.zeros_like(probs)

    for row, config in enumerate(configs):
        if config.mixture == "full":
            weights[row] = probs[row]
        elif config.mixture == GREEDY:
            weights[row, torch.argmax(probs[row])] = 1.0
        else:
            top = torch.topk(probs[row], min(mixture_top_k, probs.shape[-1]))
            total = top.values.sum()
            # A degenerate distribution can put ~0 mass in the top k only if the
            # temperature has flattened it past float resolution; fall back to
            # uniform over those k rather than dividing by zero.
            if float(total) > 0:
                weights[row, top.indices] = top.values / total
            else:
                weights[row, top.indices] = 1.0 / top.indices.numel()

    return weights


def last_position_logits(model, logits_arg, *, input_ids=None, inputs_embeds=None):
    """Run the model over a context and return the final position's logits.

    Takes either form of input. Both are needed: the decoding loop feeds
    embeddings, and the calibration below has to compare that path against the
    token-id path the model was actually trained on.

    Factored out because the rows do not all share a context length either - a
    ``last-only`` row is one position wide while the others keep growing, and a
    batch has exactly one length.
    """
    kwargs = {"use_cache": False}
    if logits_arg is not None:
        kwargs[logits_arg] = 1

    given = input_ids if inputs_embeds is None else inputs_embeds
    if inputs_embeds is None:
        kwargs["input_ids"] = input_ids
    else:
        kwargs["inputs_embeds"] = inputs_embeds

    outputs = model(
        attention_mask=torch.ones(given.shape[:2], device=given.device, dtype=torch.long),
        **kwargs,
    )
    # .clone() matters: `[:, -1, :]` is a view onto the full logits buffer, so
    # without it `del outputs` would free nothing at all.
    logits = outputs.logits[:, -1, :].to(torch.float32).clone()
    del outputs
    return logits


def scale_candidates(config):
    """Scalars worth trying when the two input paths disagree, model's own first.

    Drawn from the config rather than from a table of model ids, so a family
    this notebook has never seen contributes its own candidate. Every one is
    verified before use - nothing here is trusted because of its name.
    """
    candidates = [1.0]
    holders = [config, getattr(config, "text_config", None)]
    for holder in holders:
        if holder is None:
            continue
        for attribute in ("embedding_multiplier", "embed_scale", "scale_emb"):
            value = getattr(holder, attribute, None)
            if isinstance(value, (int, float)) and not isinstance(value, bool) and value > 0:
                candidates.append(float(value))
        hidden = getattr(holder, "hidden_size", None)
        if isinstance(hidden, int) and hidden > 0:
            candidates.append(float(hidden) ** 0.5)

    unique = []
    for candidate in candidates:
        if not any(abs(candidate - seen) < 1e-9 for seen in unique):
            unique.append(candidate)
    return unique


def calibrate_embedding_scale(model, probe_ids, logits_arg, *, tolerance=0.02):
    """Find the factor that makes feeding embeddings equal feeding token ids.

    Architectures disagree about where an embedding multiplier is applied, and
    the disagreement is invisible - no error is raised either way:

    - ``FalconH1`` multiplies inside the ``inputs_embeds is None`` branch, so
      passing embeddings skips the multiplier entirely (its own is 5.657).
    - ``GraniteMoeHybrid`` multiplies after that branch, so passing embeddings
      gets the multiplier applied on top of them (its own is 12).
    - ``Gemma`` multiplies inside the embedding module itself.
    - Most models do not multiply at all.

    Encoding those conventions would be a table of model ids that goes stale.
    Instead this asks the model: run the probe both ways and keep the scalar
    that reconciles them. If none does, raise - a decode that silently runs at
    the wrong input scale is worse than one that refuses to start.

    Returns ``(scale, errors)``; the caller feeds ``table[ids] * scale``.
    """
    table = model.get_input_embeddings().weight
    reference = last_position_logits(model, logits_arg, input_ids=probe_ids)
    magnitude = reference.abs().max().clamp_min(1e-6)

    errors = {}
    for candidate in scale_candidates(model.config):
        probe = table[probe_ids] * candidate
        logits = last_position_logits(model, logits_arg, inputs_embeds=probe)
        error = float((logits - reference).abs().max() / magnitude)
        errors[candidate] = error
        if error <= tolerance:
            return candidate, errors

    raise RuntimeError(
        "cannot reconcile the inputs_embeds path with the input_ids path for "
        f"{type(model).__name__}. Tried {errors} (relative max logit error, "
        f"tolerance {tolerance}). This model scales or transforms its input "
        "embeddings in a way none of the candidates reproduces, so feeding "
        "mixtures would run it at the wrong input scale. Add the right scalar "
        "to scale_candidates() rather than lowering the tolerance."
    )


def mix_embeddings(weights, embedding_weight):
    """The weighted sum itself: (configs, vocab) @ (vocab, hidden) -> (configs, hidden).

    Done in the embedding table's own dtype rather than float32. Materialising a
    float32 copy of the table would cost gigabytes for a 150k vocabulary, and
    the matmul accumulates in float32 on the GPU regardless; what is lost is
    precision on the individual tiny weights, whose contribution is tiny by
    construction.
    """
    return (weights.to(embedding_weight.dtype) @ embedding_weight).to(embedding_weight.dtype)


def rescale_rows(fed, configs, target_norm):
    """Scale the rows whose configuration asks for it to the mean embedding norm.

    A convex combination is shorter than the vectors it averages - more so the
    flatter the distribution - so a raw mixed vector is off-distribution in
    magnitude as well as direction. Rescaling separates the two.
    """
    wants = torch.tensor([config.rescale for config in configs], device=fed.device)
    norms = fed.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    scaled = fed * (target_norm / norms)
    return torch.where(wants.unsqueeze(-1), scaled, fed)


def nearest_token(fed, embedding_weight, embedding_norms):
    """Cosine similarity to the closest real token embedding, and which one.

    The headline diagnostic: it says how far outside the vocabulary the fed
    vector has drifted. 1.0 means it is pointing exactly at a real token; a
    mixture that stays near 1.0 is barely mixing at all.
    """
    unit = fed / fed.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    sims = (unit.to(embedding_weight.dtype) @ embedding_weight.T).to(torch.float32)
    sims /= embedding_norms.clamp_min(1e-6)
    best = sims.max(dim=-1)
    return best.values, best.indices

### 4c. Implementation, part 3 of 3

In [ ]:
@torch.inference_mode()             # no autograd graph, no gradient buffers
def weighted_embedding_decode(model, tokenizer, prompt, configs, *,
                              num_tokens_to_generate=100, record_top_k=10,
                              mixture_top_k=5, device=None):
    """Decode every configuration in lockstep, feeding back mixed embeddings.

    One batch row per configuration. They share the prompt and the step index,
    so a difference between two rows at step t is caused by the configuration
    and nothing else - which is the only reason the greedy control is in here
    rather than in a run of its own.
    """
    validate_configs(configs, num_tokens_to_generate, record_top_k, mixture_top_k)
    settings = dict(num_tokens_to_generate=num_tokens_to_generate,
                    record_top_k=record_top_k, mixture_top_k=mixture_top_k,
                    device=device)

    model.eval()          # a live dropout would silently corrupt every probability
    device = resolve_device(model, device)
    logits_arg = logits_to_keep_arg(model)
    eos_ids = eos_token_ids(model, tokenizer)

    # The portable accessor, not a memorised attribute name: the seven families
    # here disagree about what the embedding module is called.
    embeddings = model.get_input_embeddings()
    embedding_weight = embeddings.weight
    embedding_norms = embedding_weight.to(torch.float32).norm(dim=-1)
    target_norm = embedding_norms.mean()

    prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)
    if prompt_ids.shape[1] == 0:
        raise ValueError("the prompt tokenised to nothing; decoding needs a context")

    # Everything fed to the model is `embedding_weight[ids] * scale`, with the
    # scale measured against this model rather than assumed. Indexing the table
    # directly rather than calling the embedding module matters: some modules
    # apply a multiplier of their own, and going through them would apply it to
    # the prompt but not to a mixture, which is the same bug in a new place.
    scale, scale_errors = calibrate_embedding_scale(model, prompt_ids, logits_arg)
    settings["embedding_scale"] = scale

    # The prompt is embedded through the model's own embedding module, so the
    # prompt rows and the mixed rows land in exactly the same space - including
    # any scaling the architecture applies on the way in.
    rows = len(configs)
    prompt_embeds = embedding_weight[prompt_ids] * scale
    temperatures = torch.tensor([c.temperature for c in configs],
                                device=device, dtype=torch.float32).unsqueeze(-1)

    # Rows whose history is LAST_ONLY keep a one-position context once the
    # prompt step is behind them, so they cannot share a tensor with the growing
    # rows - a batch has exactly one length. They are run as a second, tiny
    # forward pass rather than by masking the growing tensor: masking is an
    # attention concept, and on the hybrid SSM models here a masked position
    # still runs through the recurrence, which is not the same as not being
    # there. The extra pass is one position wide and costs almost nothing.
    growing_rows = [i for i, c in enumerate(configs) if c.history != LAST_ONLY]
    window_rows = [i for i, c in enumerate(configs) if c.history == LAST_ONLY]

    growing_ctx = prompt_embeds.repeat(len(growing_rows), 1, 1) if growing_rows else None
    window_ctx = prompt_embeds.repeat(len(window_rows), 1, 1) if window_rows else None

    steps, top_tokens = [], []
    readouts = {config.name: [] for config in configs}
    wants_commit = torch.tensor([c.history == COMMITTED for c in configs], device=device)
    previous_argmax = None      # what the position we are about to leave behind named

    for step in range(1, num_tokens_to_generate + 1):
        # One pass per context group, then the rows are put back in order so
        # everything downstream sees a single (rows, vocab) tensor.
        parts = []
        if growing_rows:
            parts.append((growing_rows,
                          last_position_logits(model, logits_arg,
                                               inputs_embeds=growing_ctx)))
        if window_rows:
            parts.append((window_rows,
                          last_position_logits(model, logits_arg,
                                               inputs_embeds=window_ctx)))

        logits = torch.empty((rows, parts[0][1].shape[-1]),
                             device=device, dtype=torch.float32)
        for where, part in parts:
            logits[where] = part
        del parts

        logits /= temperatures                  # each row at its own temperature
        log_probs = torch.log_softmax(logits, dim=-1)
        probs = log_probs.exp()
        del logits

        entropy = -(probs * log_probs).sum(dim=-1)
        recorded = torch.topk(probs, min(record_top_k, probs.shape[-1]), dim=-1)
        argmax_ids = recorded.indices[:, 0]

        weights = mixture_weights(probs, configs, mixture_top_k)
        # The mixing matmul is the expensive part of the step - (rows, vocab) by
        # (vocab, hidden). Compute it once and take the unscaled norm from the
        # same tensor rather than mixing a second time to measure it.
        mixed = mix_embeddings(weights, embedding_weight)
        raw_norms = mixed.norm(dim=-1)
        fed = rescale_rows(mixed, configs, target_norm)
        cosine, nearest_ids = nearest_token(fed, embedding_weight, embedding_norms)
        fed_norms = fed.norm(dim=-1)
        # Diagnostics are measured before the scale is applied, against the raw
        # table they are compared to. The scale is one constant applied to real
        # tokens and mixtures alike, so norm_ratio and cosine are unaffected by
        # it; measuring pre-scale just keeps them comparable across models.
        scaled_fed = fed * scale
        top_mass = recorded.values.sum(dim=-1)

        for row, config in enumerate(configs):
            token_id = int(argmax_ids[row])
            readouts[config.name].append(token_id)
            steps.append({
                "config": config.name,
                "temperature": config.temperature,
                "mixture": config.mixture,
                "rescaled": config.rescale,
                "history": config.history,
                "step": step,
                "entropy": float(entropy[row]),
                "fed_norm": float(fed_norms[row]),
                "unscaled_norm": float(raw_norms[row]),
                "norm_ratio": float(fed_norms[row] / target_norm),
                "cosine_to_nearest": float(cosine[row]),
                "nearest_token": tokenizer.decode([int(nearest_ids[row])]),
                "top_k_mass": float(top_mass[row]),
                "argmax_token_id": token_id,
                "argmax_token": tokenizer.decode([token_id]),
                "argmax_probability": float(recorded.values[row, 0]),
                # Recorded, never acted on - the loop always runs the full budget.
                "argmax_is_eos": token_id in eos_ids,
            })
            for rank in range(recorded.indices.shape[-1]):
                candidate = int(recorded.indices[row, rank])
                top_tokens.append({
                    "config": config.name,
                    "step": step,
                    "rank": rank + 1,
                    "token_id": candidate,
                    "token": tokenizer.decode([candidate]),
                    "probability": float(recorded.values[row, rank]),
                })

        # Commit the position the loop is about to leave behind, for the rows
        # that ask for a hard history. It has already served its purpose - the
        # distribution just computed was conditioned on it - so replacing it now
        # with the embedding of the token it named leaves exactly one continuous
        # position in the sequence, the newest, for those rows. Rows with a soft
        # history keep every mixture they ever produced.
        if growing_rows:
            if previous_argmax is not None and bool(wants_commit.any()):
                hard = embedding_weight[previous_argmax[growing_rows]] * scale
                growing_ctx[:, -1, :] = torch.where(
                    wants_commit[growing_rows].unsqueeze(-1), hard,
                    growing_ctx[:, -1, :])
            growing_ctx = torch.cat(
                [growing_ctx, scaled_fed[growing_rows].unsqueeze(1)], dim=1)

        # A last-only row's whole context is replaced, not appended to: the
        # newest mixture is all it will ever see next step.
        if window_rows:
            window_ctx = scaled_fed[window_rows].unsqueeze(1)

        previous_argmax = argmax_ids
        del probs, log_probs, weights, mixed, fed, scaled_fed

    result = WeightedEmbeddingResult(
        prompt=prompt, prompt_token_ids=prompt_ids[0].tolist(), config=settings)
    result.configs = configs
    result.steps = steps
    result.top_tokens = top_tokens
    result.readouts = readouts
    result.embedding_scale = scale
    result.scale_errors = scale_errors
    return result


def divergence_step(readouts, control=GREEDY):
    """First step at which each row's argmax stops matching the control's.

    Free-running, not teacher-forced: once a row diverges its prefix differs, so
    this is "when did it leave", not a per-step disagreement rate. `None` means
    it never left.
    """
    reference = readouts[control]
    diverged = {}
    for name, tokens in readouts.items():
        if name == control:
            continue
        step = next((i + 1 for i, (a, b) in enumerate(zip(tokens, reference, strict=True))
                     if a != b), None)
        diverged[name] = step
    return diverged

## 5. One model, end to end

Download, load, decode every configuration in one batch, save, free, delete.
The saves happen **before** anything is released, so a failure on a later model
cannot cost the results of an earlier one - and weight deletion happens in a
`finally`, including after a download that failed part-way.

In [ ]:
def run_one(spec):
    """Download, load, decode, save, free, delete. Returns one summary row."""
    model_id = spec["id"]
    out = results_dir(model_id)

    if spec["bf16_GiB"] > VRAM_BUDGET_GIB:
        raise MemoryError(
            f"needs {spec['bf16_GiB']:.1f} GiB of weights but the budget is "
            f"{VRAM_BUDGET_GIB:.1f} GiB on this {GPU_NAME}"
        )

    needs_code = spec.get("remote_code", False)
    if needs_code and not TRUST_REMOTE_CODE:
        raise PermissionError(
            f"{model_id} ships its own modelling code and cannot load without "
            f"trust_remote_code; set TRUST_REMOTE_CODE = True in section 3 if "
            f"you have decided to trust this repo"
        )

    print(f"  disk free {disk_free_gib():.0f} GiB - downloading", flush=True)
    t0 = time.perf_counter()
    download(model_id)
    download_seconds = time.perf_counter() - t0
    print(f"  downloaded in {download_seconds:.0f}s", flush=True)

    t0 = time.perf_counter()
    model, tokenizer = load(model_id, trust=needs_code)
    load_seconds = time.perf_counter() - t0
    vram = torch.cuda.memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    print(f"  loaded in {load_seconds:.0f}s, {vram:.1f} GiB on GPU", flush=True)

    try:
        prompt, used_template = build_prompt(tokenizer, PROMPT)
        print(f"  chat template: {'applied' if used_template else 'none - raw completion'}",
              flush=True)

        configs = build_configs(TEMPERATURES, MIXTURES, MAGNITUDES, HISTORIES)
        print(f"  decoding {len(configs)} rows in one batch", flush=True)

        t0 = time.perf_counter()
        result = weighted_embedding_decode(model, tokenizer, prompt, configs, **CONFIG)
        decode_seconds = time.perf_counter() - t0
        print(f"  decode {decode_seconds:.0f}s, "
              f"embedding scale {result.embedding_scale:.4g}", flush=True)

        steps_df = pd.DataFrame(result.steps)
        top_df = pd.DataFrame(result.top_tokens)
        diverged = divergence_step(result.readouts)

        # --- save, named for the model, before anything is freed -------------
        steps_df.to_csv(out / "per-step-diagnostics.csv", index=False)
        top_df.to_csv(out / "top-token-probabilities.csv", index=False)

        readout_rows = [
            {
                "config": name,
                "diverged_from_greedy_at": diverged.get(name),
                "text": tokenizer.decode(tokens),
                "token_ids": " ".join(str(t) for t in tokens),
            }
            for name, tokens in result.readouts.items()
        ]
        pd.DataFrame(readout_rows).to_csv(out / "argmax-readouts.csv", index=False)

        (out / "run-metadata.json").write_text(json.dumps({
            "model": model_id, "family": spec["family"],
            "instruction": PROMPT,          # what was asked
            "prompt": prompt,               # what the model was actually given
            "used_chat_template": used_template,
            "embedding_scale": result.embedding_scale,
            "embedding_scale_errors": {str(k): v for k, v in result.scale_errors.items()},
            "dtype": str(DTYPE), "device": DEVICE, "gpu": GPU_NAME, "seed": SEED,
            "config": dict(CONFIG),
            "temperatures": TEMPERATURES,
            "mixtures": MIXTURES,
            "magnitudes": MAGNITUDES,
            "histories": HISTORIES,
            "download_s": round(download_seconds, 1),
            "load_s": round(load_seconds, 1),
            "decode_s": round(decode_seconds, 1),
        }, indent=2))

        soft = steps_df[steps_df["mixture"] != GREEDY]
        row = {
            "model": model_id,
            "family": spec["family"],
            "params_B": spec["params_B"],
            "bf16_GiB": spec["bf16_GiB"],
            "used_chat_template": used_template,
            "embedding_scale": result.embedding_scale,
            "blocks": len(block_names(model)),
            "vocab": vocab_size(model),
            "gpu_GiB": round(vram, 2),
            "download_s": round(download_seconds, 1),
            "load_s": round(load_seconds, 1),
            "decode_s": round(decode_seconds, 1),
            "rows": len(configs),
            # How far outside the vocabulary the fed vectors drifted, and how
            # quickly the readouts left the control's path.
            "mean_cosine_to_nearest": round(float(soft["cosine_to_nearest"].mean()), 4),
            "min_cosine_to_nearest": round(float(soft["cosine_to_nearest"].min()), 4),
            "mean_norm_ratio": round(float(soft["norm_ratio"].mean()), 4),
            "mean_entropy": round(float(soft["entropy"].mean()), 4),
            "mean_cosine_soft_history": round(
                float(soft[soft["history"] == SOFT]["cosine_to_nearest"].mean()), 4),
            "mean_cosine_committed_history": round(
                float(soft[soft["history"] == COMMITTED]["cosine_to_nearest"].mean()), 4),
            "mean_cosine_last_only_history": round(
                float(soft[soft["history"] == LAST_ONLY]["cosine_to_nearest"].mean()), 4),
            "median_divergence_step": (
                pd.Series([v for v in diverged.values() if v is not None]).median()
                if any(v is not None for v in diverged.values()) else None
            ),
            "never_diverged": sum(1 for v in diverged.values() if v is None),
            "greedy_text": tokenizer.decode(result.readouts[GREEDY]),
        }
        # The summary row goes to disk too, so a resumed run can rebuild the
        # cross-model table from models it finished in an earlier session.
        (out / "summary-row.json").write_text(json.dumps(row, indent=2))
        return row
    finally:
        try:
            del model, tokenizer
        except NameError:
            pass
        free_gpu()

## 6. Run the sweep

> **The long cell.** Each model downloads (13-29 GiB), loads, then runs 100
> batched forward passes over 21 rows. Download time dominates.
>
> A failure in one model is recorded and the sweep continues; skips land in
> `skipped-models.csv` with the reason.
>
> **This cell is resumable.** If the runtime disconnects, reconnect, re-run
> sections 0-6, then re-run this cell: every model whose `summary-row.json` is
> already on Drive is skipped.

In [ ]:
# Re-running this cell after a disconnect picks up where it stopped: a model
# whose summary-row.json is already on Drive is not downloaded or run again.
# Set RERUN_COMPLETED = True to force the whole sweep from scratch.
RERUN_COMPLETED = False

failures = {}

for spec in MODELS:
    model_id = spec["id"]
    done = results_dir(model_id) / "summary-row.json"

    if done.exists() and not RERUN_COMPLETED:
        print(f"\n=== {model_id}  - already done, skipping", flush=True)
        continue

    print(f"\n=== {model_id}  ({spec['bf16_GiB']} GiB)", flush=True)
    try:
        row = run_one(spec)
        print(f"  greedy:   {row['greedy_text'][:90]!r}", flush=True)
        print(f"  cosine to nearest token, mean over soft rows: "
              f"{row['mean_cosine_to_nearest']}", flush=True)
    except Exception as exc:  # noqa: BLE001 - one model must not end the sweep
        failures[model_id] = f"{type(exc).__name__}: {exc}"
        print(f"  SKIPPED - {failures[model_id]}", flush=True)
    finally:
        # Always reclaim the disk, including after a failure part-way through a
        # download - otherwise a later model has nowhere to land.
        delete_weights(model_id)
        print(f"  weights removed, disk free {disk_free_gib():.0f} GiB", flush=True)

order = {spec["id"]: i for i, spec in enumerate(MODELS)}
rows = [json.loads((results_dir(s["id"]) / "summary-row.json").read_text())
        for s in MODELS if (results_dir(s["id"]) / "summary-row.json").exists()]
summary_df = pd.DataFrame(sorted(rows, key=lambda r: order.get(r["model"], 99)))

print(f"\n{len(summary_df)} of {len(MODELS)} models have results on disk")
for model_id, why in failures.items():
    print(f"  failed this session: {model_id} - {why}")

## 7. Results

Same prompt, same budget, same seed throughout, so every difference belongs to
the model or the configuration.

`cosine_to_nearest` is the one to watch: it is how far the fed vector has
drifted outside the vocabulary. Near 1.0 means the mixture is dominated by a
single token and the run is greedy in disguise; low means the model is being
fed something no token could have produced.

In [ ]:
summary_df[[
    "model", "family", "mean_cosine_soft_history", "mean_cosine_committed_history",
    "mean_cosine_last_only_history", "mean_norm_ratio", "mean_entropy",
    "median_divergence_step", "never_diverged",
]]

## 8. Export

`<model-slug>/` per model, plus a cross-model summary and, if anything was
skipped, the reasons. These are already on Drive; the zip is for the trip back
to the repo, where the folder names drop straight into `results/`.

The last cell prints the local command to run once the download lands.

In [ ]:
summary_path = RESULTS / "weighted-embedding-feedback.csv"
summary_df.to_csv(summary_path, index=False)

if failures:
    pd.DataFrame(
        [{"model": k, "reason": v} for k, v in failures.items()]
    ).to_csv(RESULTS / "skipped-models.csv", index=False)

archive = shutil.make_archive(f"/content/{EXPERIMENT}-results", "zip", RESULTS)
print(f"{archive}  ({Path(archive).stat().st_size / 1e6:.1f} MB)")

for path in sorted(RESULTS.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(RESULTS)}")

In [ ]:
# Colab cannot write to your machine: files.download() pushes the file to the
# browser's download folder and that is where its reach ends.
from google.colab import files

files.download(archive)

name = Path(archive).name
print()
print(f"{name} is in your browser's download folder.")

## Findings

Write these up in `docs/` in the repo - a result that lives only in a Colab
session that will be recycled is not finished.

Each model's `run-metadata.json` records what is needed to reproduce it: model
id, dtype, device, GPU, seed, the full `CONFIG` and all three axes.

The comparison this exists to support is against
`naive-beam-search-colab.ipynb`, which shares the prompt exactly. Both must be
run with the *same* prompt for that comparison to mean anything - if you change
one, re-run the other.